# Crop Disease Detection — Model Experiments
**Team Winters (T-66) | GLA University**

This notebook covers:
1. Dataset exploration
2. Model training and evaluation
3. Confusion matrix
4. Inference speed benchmark

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 1. Dataset Exploration

In [ ]:
DATASET_PATH = '../data/processed/'

# Count images per class
classes = sorted(os.listdir(DATASET_PATH))
counts  = {cls: len(os.listdir(os.path.join(DATASET_PATH, cls))) for cls in classes if os.path.isdir(os.path.join(DATASET_PATH, cls))}

print(f'Total classes : {len(counts)}')
print(f'Total images  : {sum(counts.values())}')
print()
for cls, n in counts.items():
    print(f'  {cls:<50} {n:>5} images')

In [ ]:
# Plot class distribution
plt.figure(figsize=(16, 6))
plt.barh(list(counts.keys()), list(counts.values()), color='#4caf50')
plt.xlabel('Number of Images')
plt.title('Images per Disease Class')
plt.tight_layout()
plt.savefig('../docs/class_distribution.png', dpi=100)
plt.show()

## 2. Load Data Generators

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, validation_split=0.2
).flow_from_directory(DATASET_PATH, target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical', subset='training')

val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2
).flow_from_directory(DATASET_PATH, target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', shuffle=False)

print(f'Training:   {train_gen.samples} images, {train_gen.num_classes} classes')
print(f'Validation: {val_gen.samples} images')

## 3. Build & Train Model

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

base  = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('../models/disease_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

history = model.fit(train_gen, epochs=20, validation_data=val_gen, callbacks=callbacks)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout(); plt.savefig('../models/training_plot.png', dpi=100); plt.show()

## 5. Confusion Matrix

In [ ]:
val_gen.reset()
preds      = model.predict(val_gen, verbose=1)
y_pred     = np.argmax(preds, axis=1)
y_true     = val_gen.classes
class_names = list(val_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Greens')
plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.savefig('../docs/confusion_matrix.png', dpi=100); plt.show()

## 6. TFLite Conversion & Speed Benchmark

In [ ]:
import time

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('../models/disease_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model size: {len(tflite_model) / 1024:.1f} KB')

# Benchmark
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()

dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
interpreter.set_tensor(inp[0]['index'], dummy)

times = []
for _ in range(20):
    t0 = time.perf_counter()
    interpreter.invoke()
    times.append((time.perf_counter() - t0) * 1000)

print(f'Mean inference : {np.mean(times):.1f} ms')
print(f'p95  inference : {np.percentile(times, 95):.1f} ms')
print(f'Min  inference : {np.min(times):.1f} ms')